# ⚡ Notebook 03: Negation-Aware Deterministic Crisis Router (<5ms SLA)
### SENTINEL V4 Research Lab · Build for Bharat 2026 (PS 3: AI for Citizen Impact)
**Objective:** Deterministic routing for Tier 1 crisis detection with false-alarm negation handling across 4 languages.


In [ ]:
import sys
import os
import time

sys.path.insert(0, os.path.abspath(".."))

from models.crisis_router import get_crisis_router

router = get_crisis_router()
print("Loaded CrisisRouter Engine.")


## 1. Safe Negation Benchmark
A common failure of LLM/Keyword triage is flagging safe negations such as:
- *"I am stressed about exams, but I don't want to die."*
- *"mujhe marna nahi hai, bas guide chahiye"*
SENTINEL uses negative lookbehind/lookahead regular automata to cleanly suppress false alarms in under 0.1ms.


In [ ]:
test_sentences = [
    ("I'm feeling stressed about exams, but I don't want to die.", False, "Safe Negation (English)"),
    ("sab khatam karna hai tonight, cannot take it anymore", True, "Acute Crisis (Hinglish)"),
    ("I am definitely not suicidal, just tired", False, "Safe Negation (English)"),
    ("ab aur nahi sah sakta, I want to end tonight", True, "Acute Crisis (Hindi/Hinglish)"),
    ("mujhe marna nahi hai, bas career ka tension hai", False, "Safe Negation (Hindi)"),
    ("vaazhkai mudikkanum, vali thaanga mudiyala", True, "Acute Crisis (Tamil)")
]

for text, should_flag, label in test_sentences:
    t0 = time.perf_counter()
    res = router.route(text)
    dt = (time.perf_counter() - t0) * 1000.0
    status = "ALERT TRIGGERED" if res.is_acute_crisis else "CLEAR (PASS)"
    match = res.is_acute_crisis == should_flag
    print(f"[{'PASS' if match else 'FAIL'}] {dt:.3f}ms | {status:<15} | {label} | '{text}'")


## 2. High-Throughput Latency Stress Test (10,000 Queries)


In [ ]:
N = 10000
phrases = [s[0] for s in test_sentences]

t_start = time.perf_counter()
for i in range(N):
    router.route(phrases[i % len(phrases)])
t_total = time.perf_counter() - t_start
avg_ms = (t_total * 1000.0) / N

print(f"Total Iterations:    {N:,}")
print(f"Total Elapsed Time:  {t_total:.4f} s")
print(f"Average Latency:     {avg_ms:.4f} ms/query")
print(f"Throughput:          {N / t_total:,.0f} queries/sec")
print(f"Government SLA:      < 5.0000 ms (Met: {avg_ms < 5.0})")
